# ****DATA UNDERSTANDING****

<p align="center">
    <img src="../xLain-Lain/img/du_1.png" alt="Pendahuluan" witdh="100" height="500">
</p>

### ****DAFTAR ISI****

1. Persiapan dan Pemuatan Data
2. Struktur, Tipe, dan Statistik Deskriptif
3. Audit Kalender dan Missing Value
4. Audit Stale Price
5. Verifikasi Definisi Target $Y$ dan Jebakan Kebocoran Lag
6. Audit Anomali Data
7. Frekuensi Pembaruan Variabel Makro
8. Distribusi dan Dinamika Target
9. Volatility Clustering
10. Sinyal Awal Antara Kandidat Prediktor dan Target
11. Stasioneritas
12. Pergeseran Rezim: Data Latih vs Data Uji
13. Ringkasan Temuan dan Decision Log

---

### ****1. Persiapan dan Pemuatan Data****

Sel ini memuat pustaka dan membaca data mentah. Kolom Date dibaca sebagai *datetime* dan dijadikan indeks agar operasi deret waktu (rolling, shift) berbasis tanggal. Dua konstanta penting didefinisikan di sini dan dipakai di seluruh notebook kami ini adalah:

- TRAIN_END = 2024-12-3: batas data berlabel. Seluruh baris sejak 2025 adalah data uji (tanpa $Y$).
- PRICE_COLS dan MACRO_COLS: pemisahan variabel pasar harian dan variabel makro berfrekuensi rendah.

In [1]:
# muat pustaka

import warnings, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import statsmodels
from statsmodels.tsa.stattools import acf, adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [3]:
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# sekadar informasi kalau tim kami memang sudah ada format pengerjaan dan framework tersendiri, jadinya ini boleh diabaikan karena hanya
# pendefinisian path saja... 

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, cwd.parent] if (p / "Dataset").exists()), cwd)
RAW_DIR, CLEAN_DIR = ROOT / "Dataset" / "Raw", ROOT / "Dataset" / "Clean"
IMG_DIR, NOTE_DIR = ROOT / "xLain-Lain" / "img", ROOT / "Catatan"
for d in (CLEAN_DIR, IMG_DIR, NOTE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# gaya visual

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10.5,
    "axes.titlesize": 12.5, "axes.titleweight": "bold", "axes.labelsize": 10.5,
    "axes.spines.top": False, "axes.spines.right": False,
})
BLUE, ORANGE, GREEN, RED, PURPLE, GREY = "#2F6DB5", "#E8833A", "#3E9B5F", "#C0392B", "#7B5EA7", "#8A8F98"

def save_fig(fig, name):
    fig.savefig(IMG_DIR / name, bbox_inches="tight", facecolor="white")
    print(f"Gambar disimpan: xLain-Lain/img/{name}")

# Konstanta proyek

TARGET = "Target_Volatilitas_Y"
PRICE_COLS = ["OIL", "GOLD", "USDIDR", "SP500", "IHSG", "VIX"]
MACRO_COLS = ["CPI", "BI_rate", "US_rate"]
TRAIN_END = pd.Timestamp("2024-12-31")
TEST_START = pd.Timestamp("2025-01-01")

# muat raw data

DATA_PATH = next(p for p in [RAW_DIR / "dataset peserta.csv", RAW_DIR / "dataset_peserta.csv"] if p.exists())
df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").set_index("Date")

y = df[TARGET]
is_lab = y.notna()
print(f"Berkas : {DATA_PATH.relative_to(ROOT)}")
print(f"Dimensi : {df.shape[0]:,} baris x {df.shape[1]} kolom | {df.index.min():%Y-%m-%d} s.d. {df.index.max():%Y-%m-%d}")
# print(f"pandas {pd.__version__} | numpy {np.__version__} | statsmodels {statsmodels.__version__} | matplotlib {matplotlib.__version__}")

Berkas : Dataset\Raw\dataset_peserta.csv
Dimensi : 4,266 baris x 10 kolom | 2010-01-04 s.d. 2026-05-22
